# Autoship Nudge Promo Incentive — Power Analysis (Autoship Adoption Rate, 2-Cell Design, Verified Population)

**Experiment:** Autoship Nudge Promo Incentive Test · **Owner:** Sergio Oyola · **Primary metric analyzed here:** Autoship Adoption Rate · **Randomization unit:** `client_id` · **Allocation point:** post-First-Fix checkout, once keep rate is known, at the moment the client selects a Quick Fix date and clicks "Schedule a Quick Fix" (prior to Narvar handoff)

This notebook sizes a standard 2-cell A/B version of the experiment for its primary metric, **Autoship Adoption Rate**, adding three population-integrity checks on top of the base eligibility criteria: confirming genuine first-time status, and excluding employees and fraudulent accounts.

## Population
Manual clients — i.e., not already enrolled in Autoship — who completed First Fix checkout with a **Buy 1+** keep rate (kept at least one item). Buy 0 clients always see the BAU Quick Fix experience with no Autoship nudge and are out of scope for this comparison, per the PRD. On top of these base criteria, three additional checks are applied:

1. **Genuinely first-time, not reactivated.** `fix_number = 1` on a shipment is meant to mark a client's first-ever Fix, but a client returning after a long absence could in principle show up the same way if their history isn't accounted for. This notebook cross-checks each client's state in `curated.checkout_based_client_state_journal` as of their First Fix checkout, and only keeps clients whose state was `'Never Active'` at that point — i.e., no prior checkout history of any kind, not `Engaged`, `Dormant`, or `Lapsed`.
2. **Employees excluded.** `curated.client.employee_affiliated_flag` marks employee-affiliated accounts, which are excluded from the eligible population.
3. **Fraudulent clients excluded.** `curated.client.fake_client_flag` marks accounts identified as fraudulent, which are excluded from the eligible population.

## Design: 2-cell test, single comparison
Eligible clients are randomized into 2 cells at a 50/50 split:

| Cell | Experience | Offer |
|---|---|---|
| Control | BAU Quick Fix, no Autoship nudge | None |
| Treatment | Autoship nudge + promo billboard | 10% off next eligible Fix |

A single pairwise comparison is planned: **Treatment vs. Control**, measuring the combined effect of introducing the Autoship nudge together with the promo incentive, relative to today's BAU experience. Because only one comparison is planned against the family-wise error budget, no multiple-comparison correction is needed here — sizing uses the initial `alpha = 0.05` directly.

## One-sided test
For Autoship Adoption Rate, a **flat** result (no lift from the nudge+promo experience) and a **negative** result (the nudge+promo experience underperforms BAU) lead to the identical rollout decision: do not roll out the nudge+promo experience, continue with the BAU Quick Fix flow. There is no decision on the table that requires distinguishing "no effect" from "a harmful effect" — so this sizing is **one-sided**, powered only to detect a positive lift from the nudge+promo experience over BAU.

## Metric definition
**Autoship Adoption Rate** = share of eligible clients who show a fresh Autoship demand event within a 90-day window following their First Fix checkout.

A client's Autoship history is tracked in `curated.client_pulse_journal`, a daily journal (one row per day any tracked client attribute changes) carrying `last_autoship_demand_ts` — the timestamp of that client's most recent Autoship demand event as of that journal row. A client is counted as **adopted** if, scanning their full journal history, the *earliest* `last_autoship_demand_ts` value that is itself later than their First Fix checkout date falls within 90 days of that checkout.

Two choices in this definition are deliberate:
- **Scanning the full journal history, not a single current-day snapshot.** `last_autoship_demand_ts` can reset when a client's Autoship subscription is fully cancelled; reading only today's value would silently drop clients who adopted and later cancelled. Scanning the full history for the earliest post-First-Fix value is robust to that reset.
- **Requiring the demand timestamp to be strictly *after* First Fix checkout, not merely populated.** A meaningful share of clients carry a pre-existing Autoship demand timestamp that predates their First Fix entirely (e.g., a historical Autoship enrollment unrelated to this test's post-checkout nudge). Requiring the timestamp to fall after checkout excludes that population instead of miscounting them as adopters.

**Caveat:** `last_autoship_demand_ts` reflects when a client's Autoship subscription was (re-)created, not each individual shipment under that subscription, so it is an opt-in-*adjacent* signal — one step removed from a literal "clicked opt-in" event log. It also cannot confirm that a given adoption was caused specifically by the post-First-Fix nudge, as opposed to some other Autoship enrollment channel — a limitation shared by any warehouse-derived adoption proxy, not unique to this one.

In [1]:
import numpy as np
import pandas as pd
from amphibian import get_data_accessor
from power import n_total_statsmodels

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

# Data parameters
COHORT_START = '2025-08-01'
MATURATION_DAYS = 90  # days to wait for a client's Autoship demand event to resolve

# Design parameters (2-cell test, single comparison, no multiple-comparison correction)
ALPHA = 0.05  # single comparison: Treatment vs. Control, no Bonferroni adjustment needed
POWER = 0.80
TWO_SIDED = False  # one-sided: only a positive lift over BAU changes the rollout decision
N_ARMS = 2
SPLIT = 0.5  # Control and Treatment are equal-sized arms
MDE_GRID = [0.02, 0.03, 0.04, 0.05]  # relative lift on Autoship Adoption Rate, Treatment vs. Control

## Step 1 — Autoship Adoption Rate baseline & daily eligible volume

The eligible population is identified from `curated.merch_sales_and_feedback`: each client's earliest `fix_number = 1` shipment, gated to Manual (`autoship_or_manual = 'manual'`) + Buy 1+ (at least one item kept), with a 90-day maturation cutoff so the demand-event read has had time to resolve. On top of that, `state_at_fix` confirms each client's state was `'Never Active'` immediately before that checkout (per `curated.checkout_based_client_state_journal`), and `curated.client`'s `fake_client_flag` and `employee_affiliated_flag` exclude fraudulent and employee accounts. Adoption is then read from `curated.client_pulse_journal`'s full history, per the metric definition above.

In [2]:
baseline_query = f"""--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '{COHORT_START}'
    GROUP BY client_id, shipment_id
),
first_fix_deduped AS (
    SELECT client_id, checkout_date, autoship_or_manual, n_items_kept
    FROM (
        SELECT client_id, checkout_date, autoship_or_manual, n_items_kept,
               ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date) AS rn
        FROM first_fix
    )
    WHERE rn = 1
),
state_at_fix AS (
    -- Each client's state as of the journal row covering their First Fix checkout.
    SELECT f.client_id, j.client_state_detail,
           ROW_NUMBER() OVER (PARTITION BY f.client_id ORDER BY j.start_timestamp DESC) AS rn
    FROM first_fix_deduped f
    JOIN curated.checkout_based_client_state_journal j
      ON j.client_id = f.client_id
     AND j.start_timestamp <= CAST(f.checkout_date AS TIMESTAMP)
),
eligible AS (
    SELECT f.client_id, f.checkout_date
    FROM first_fix_deduped f
    JOIN curated.client c ON c.client_id = f.client_id
    JOIN state_at_fix s ON s.client_id = f.client_id AND s.rn = 1
    WHERE f.autoship_or_manual = 'manual'
      AND f.n_items_kept >= 1
      AND f.checkout_date <= CURRENT_DATE - INTERVAL '{MATURATION_DAYS}' DAY
      AND COALESCE(c.fake_client_flag, 0) = 0
      AND COALESCE(c.employee_affiliated_flag, 0) = 0
      AND s.client_state_detail = 'Never Active'
),
fresh_demand AS (
    SELECT e.client_id, MIN(p.last_autoship_demand_ts) AS first_fresh_demand_ts
    FROM eligible e
    JOIN curated.client_pulse_journal p
      ON p.client_id = e.client_id
     AND p.last_autoship_demand_ts > e.checkout_date
    GROUP BY e.client_id
),
joined AS (
    SELECT
        e.client_id,
        DATE_TRUNC('month', e.checkout_date) AS month,
        e.checkout_date,
        CASE WHEN f.first_fresh_demand_ts IS NOT NULL
              AND f.first_fresh_demand_ts <= e.checkout_date + INTERVAL '{MATURATION_DAYS}' DAY
             THEN 1 ELSE 0 END AS adopted_autoship
    FROM eligible e
    LEFT JOIN fresh_demand f ON f.client_id = e.client_id
),
month_days AS (
    SELECT month, COUNT(DISTINCT checkout_date) AS days_observed
    FROM joined
    GROUP BY month
)
SELECT
    j.month,
    md.days_observed,
    COUNT(*) AS n_eligible,
    SUM(adopted_autoship) AS n_adopted,
    CAST(SUM(adopted_autoship) AS DOUBLE) / COUNT(*) AS autoship_adoption_rate,
    ROUND(COUNT(*) / CAST(md.days_observed AS DOUBLE), 1) AS eligible_per_day
FROM joined j
JOIN month_days md ON j.month = md.month
GROUP BY j.month, md.days_observed
ORDER BY j.month DESC
"""

baseline_df = query(baseline_query)
baseline_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,n_eligible,n_adopted,autoship_adoption_rate,eligible_per_day
0,2026-06-01,3,774,208,0.268734,258.0
1,2026-05-01,31,9132,2269,0.248467,294.6
2,2026-04-01,30,9803,2321,0.236764,326.8
3,2026-03-01,31,10313,2200,0.213323,332.7
4,2026-02-01,28,8340,1969,0.236091,297.9
5,2026-01-01,31,9790,2517,0.257099,315.8
6,2025-12-01,31,8388,2806,0.334526,270.6
7,2025-11-01,30,6862,1382,0.201399,228.7
8,2025-10-01,31,8708,1485,0.170533,280.9
9,2025-09-01,30,8771,1383,0.157679,292.4


In [3]:
# Reference month = most recent calendar month fully past the 90-day maturation cutoff as of this run.
REFERENCE_MONTH = '2026-05-01'
ref = baseline_df[baseline_df['month'].astype(str).str.startswith(REFERENCE_MONTH)].reset_index(drop=True)

BASELINE_RATE = float(ref['autoship_adoption_rate'][0])

print(f"BASELINE_RATE = {BASELINE_RATE:.4f}")
ref.T

BASELINE_RATE = 0.2485


,0
month,2026-05-01
days_observed,31
n_eligible,9132
n_adopted,2269
autoship_adoption_rate,0.248467
eligible_per_day,294.6


**Reference month choice:** the most recent month with every calendar day already past the 90-day maturation cutoff — more recent months are only partially mature (fewer of their days have had 90 days to produce an adoption read yet), so their rates aren't yet comparable to a full month. This is used for **BASELINE_RATE** only; see Step 1b for why eligible **volume** is read from a different, more recent month instead.

## Step 1b — A fresher volume read (decoupled from the maturation gate)

The eligible population (Manual, Buy 1+, First-Fix-complete, verified new, non-employee, non-fraudulent) has grown substantially in recent months as clients shift away from pre-checkout Autoship enrollment toward the post-checkout nudge flow this experiment targets — meaning a stale reference month understates the population's current daily run rate by a wide margin. Unlike the adoption-rate read, eligible volume doesn't need the 90-day maturation wait (all three population checks are known immediately at First Fix checkout), so it can be measured off the **most recent complete month** instead of the same reference month as the rate.

In [4]:
volume_query = """--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '2026-05-01'
    GROUP BY client_id, shipment_id
),
first_fix_deduped AS (
    SELECT client_id, checkout_date, autoship_or_manual, n_items_kept
    FROM (
        SELECT client_id, checkout_date, autoship_or_manual, n_items_kept,
               ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date) AS rn
        FROM first_fix
    )
    WHERE rn = 1
),
state_at_fix AS (
    SELECT f.client_id, j.client_state_detail,
           ROW_NUMBER() OVER (PARTITION BY f.client_id ORDER BY j.start_timestamp DESC) AS rn
    FROM first_fix_deduped f
    JOIN curated.checkout_based_client_state_journal j
      ON j.client_id = f.client_id
     AND j.start_timestamp <= CAST(f.checkout_date AS TIMESTAMP)
),
eligible AS (
    SELECT f.client_id, f.checkout_date
    FROM first_fix_deduped f
    JOIN curated.client c ON c.client_id = f.client_id
    JOIN state_at_fix s ON s.client_id = f.client_id AND s.rn = 1
    WHERE f.autoship_or_manual = 'manual'
      AND f.n_items_kept >= 1
      AND COALESCE(c.fake_client_flag, 0) = 0
      AND COALESCE(c.employee_affiliated_flag, 0) = 0
      AND s.client_state_detail = 'Never Active'
),
month_days AS (
    SELECT DATE_TRUNC('month', checkout_date) AS month, COUNT(DISTINCT checkout_date) AS days_observed
    FROM eligible
    GROUP BY 1
)
SELECT
    DATE_TRUNC('month', e.checkout_date) AS month,
    md.days_observed,
    COUNT(*) AS n_eligible,
    ROUND(COUNT(*) / CAST(md.days_observed AS DOUBLE), 1) AS eligible_per_day
FROM eligible e
JOIN month_days md ON DATE_TRUNC('month', e.checkout_date) = md.month
GROUP BY DATE_TRUNC('month', e.checkout_date), md.days_observed
ORDER BY 1 DESC
"""

volume_df = query(volume_query)
volume_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,n_eligible,eligible_per_day
0,2026-09-01,1,205,205.0
1,2026-08-01,31,24757,798.6
2,2026-07-01,31,16925,546.0
3,2026-06-01,30,11581,386.0
4,2026-05-01,31,6504,209.8
5,2026-04-01,1,10,10.0


In [5]:
# Volume reference month = most recent *complete* calendar month (no maturation gate needed for volume).
VOLUME_REFERENCE_MONTH = '2026-08-01'
vol_ref = volume_df[volume_df['month'].astype(str).str.startswith(VOLUME_REFERENCE_MONTH)].reset_index(drop=True)

DAILY_ELIGIBLE = float(vol_ref['eligible_per_day'][0])

print(f"BASELINE_RATE ({REFERENCE_MONTH[:7]}, matured) = {BASELINE_RATE:.4f}  |  DAILY_ELIGIBLE ({VOLUME_REFERENCE_MONTH[:7]}, current run rate) = {DAILY_ELIGIBLE:,.1f}")

BASELINE_RATE (2026-05, matured) = 0.2485  |  DAILY_ELIGIBLE (2026-08, current run rate) = 798.6


**Reading this:** eligible volume grows substantially between the matured reference month and the most recent complete month as the Manual + Buy 1+ First-Fix population becomes a larger share of all First Fixes. Step 2 and Step 3 below use this more recent month's volume for duration math, paired with the matured month's rate for the proportions math — the two are measured over different windows deliberately, since each is gated by a different constraint (rate needs 90-day maturation; volume does not).

## Step 2 — Sample size & duration

`n_total_statsmodels` sizes a single pairwise 50/50 comparison (Treatment vs. Control); `n_treatment` is read as the **per-arm** requirement. Each of the 2 arms accrues `DAILY_ELIGIBLE / 2` eligible clients per day under the 50/50 split, so `days_required = n_per_arm / (DAILY_ELIGIBLE / 2)`.

In [6]:
def size_table(rel_grid, baseline, daily):
    raw = n_total_statsmodels(
        baseline_rate=baseline, mde_relative=rel_grid, split_ratio=[SPLIT],
        alpha=ALPHA, power=POWER, two_sided=TWO_SIDED,
    )
    df = pd.DataFrame(raw).T.reset_index(drop=True)
    df['rel_effect'] = df['mde_relative'].apply(lambda x: f"{x:+.0%}")
    df['n_per_arm'] = df['n_treatment'].astype(int)
    df['n_total_2arm'] = df['n_per_arm'] * N_ARMS
    df['days_required'] = np.ceil(df['n_per_arm'] / (daily / N_ARMS)).astype(int)
    df['weeks_required'] = (df['days_required'] / 7).round(1)
    return df[['rel_effect', 'p_treatment', 'n_per_arm', 'n_total_2arm', 'days_required', 'weeks_required']]

sided = 'one-sided' if not TWO_SIDED else 'two-sided'
print(f"--- Autoship Adoption Rate, Treatment vs. Control (baseline={BASELINE_RATE:.1%}, alpha={ALPHA}, power={POWER:.0%}, {sided}, 50/50 split) ---")
size_table(MDE_GRID, BASELINE_RATE, DAILY_ELIGIBLE)

--- Autoship Adoption Rate, Treatment vs. Control (baseline=24.8%, alpha=0.05, power=80%, one-sided, 50/50 split) ---


,rel_effect,p_treatment,n_per_arm,n_total_2arm,days_required,weeks_required
0,+2%,0.253436,94124,188248,236,33.7
1,+3%,0.255921,41970,83940,106,15.1
2,+4%,0.258406,23685,47370,60,8.6
3,+5%,0.26089,15207,30414,39,5.6


**Reading this:** at single-digit relative MDEs, required runtime is still long, but meaningfully shorter than a 3-cell version of this same design would need — dropping to 2 cells removes the Bonferroni correction (alpha moves from 0.025 back to 0.05, requiring fewer samples per arm for the same power) and gives each arm half the daily traffic instead of a third (faster accrual per arm). Both effects compound in the same direction. No harm/guardrail grid is computed here: sizing for a one-sided positive MDE does not symmetrically size for detecting harm, and downside risk on this metric is out of scope for this sizing exercise (margin risk is covered qualitatively via the Experiment Design doc's Risk section and Decision Matrix, not powered here).

## Step 3 — Summary for the Experiment Design doc

Headline MDE below is a **placeholder 4% relative lift** on Autoship Adoption Rate (Treatment vs. Control) — the second-largest value in the Step 2 grid. Every value in this grid runs several weeks or longer (Step 2).

In [7]:
TARGET_REL_MDE = 0.04  # placeholder: 4% relative lift on Autoship Adoption Rate, Treatment vs. Control (second-largest value in the Step 2 grid)

res = n_total_statsmodels(
    baseline_rate=BASELINE_RATE,
    mde_relative=[TARGET_REL_MDE],
    split_ratio=[SPLIT],
    alpha=ALPHA,
    power=POWER,
    two_sided=TWO_SIDED,
)

n_per_arm = int(list(res.values())[0]['n_treatment'])
duration_days = int(np.ceil(n_per_arm / (DAILY_ELIGIBLE / N_ARMS)))

summary = {
    'Metric Used': 'Autoship Adoption Rate (Treatment vs. Control)',
    'Population': 'Manual clients, First Fix checkout complete, Buy 1+ keep rate, not already enrolled in Autoship, verified Never Active prior to First Fix, excluding employees and fraudulent clients',
    'Baseline Value': f"{BASELINE_RATE:.1%} ({REFERENCE_MONTH[:7]}, {MATURATION_DAYS}-day matured cohort)",
    'Daily Eligible Volume': f"{DAILY_ELIGIBLE:,.0f} / day ({VOLUME_REFERENCE_MONTH[:7]}, current run rate)",
    'Minimum Detectable Effect': f"+{TARGET_REL_MDE:.0%} relative ({BASELINE_RATE:.3f} -> {BASELINE_RATE*(1+TARGET_REL_MDE):.3f})",
    'One/Two-Sided Test': 'One-sided',
    'Significance Level': f"{ALPHA} (single comparison, no multiple-comparison correction)",
    'Statistical Power': f"{POWER:.0%}",
    'Variant Split %': '50% / 50% (Control / Treatment)',
    'Minimum Samples by Variant': f"{n_per_arm:,}",
    'Minimum Samples total (2 arms)': f"{n_per_arm*N_ARMS:,}",
    'Shortest Duration Required': f"{duration_days} days (~{duration_days/7:.1f} weeks)",
}
pd.Series(summary).to_frame('value')

,value
Metric Used,Autoship Adoption Rate (Treatment vs. Control)
Population,"Manual clients, First Fix checkout complete, Buy 1+ keep rate, not already enrolled in Autoship, verified Never Active prior to First Fix, excluding employees and fraudulent clients"
Baseline Value,"24.8% (2026-05, 90-day matured cohort)"
Daily Eligible Volume,"799 / day (2026-08, current run rate)"
Minimum Detectable Effect,+4% relative (0.248 -> 0.258)
One/Two-Sided Test,One-sided
Significance Level,"0.05 (single comparison, no multiple-comparison correction)"
Statistical Power,80%
Variant Split %,50% / 50% (Control / Treatment)
Minimum Samples by Variant,"23,685"


## Bottom line

- **Three population-integrity checks are applied on top of the base eligibility criteria:** confirmed `'Never Active'` state prior to First Fix checkout (ruling out reactivated clients being mistaken for genuinely new ones), and exclusion of employee-affiliated and fraudulent accounts.
- **Autoship Adoption Rate baseline**, read as a fresh post-First-Fix Autoship demand event within 90 days, captures clients who adopt Autoship even if they later cancel before their next Fix ships, and excludes clients whose next Fix happens to be Autoship-fulfilled by a subscription that predates their First Fix.
- **Eligible daily volume has grown substantially** as the population shifts toward the post-checkout nudge flow (Step 1b), so this notebook sizes using a matured rate paired with a fresher volume read rather than a single stale reference month.
- **A 2-cell design removes the Bonferroni correction entirely** (alpha = 0.05 instead of 0.025) and doubles each arm's daily traffic share (1/2 instead of 1/3) relative to a 3-cell version of this same design — both effects reduce required runtime.
- At a **4% relative lift** (the placeholder MDE above), the experiment needs the sample size and duration shown in Step 3. Smaller MDEs (2-3%) push runtime out considerably further, while 5% is comparatively faster (Step 2).